[2026-10-16 Fiddler](https://thefiddler.substack.com/p/can-you-gerrymander-like-a-dummy)
====================

Fiddler
-------
The answer is to pack 3 districts with the opposition voters and crack 4 districts,
so the minimum needed to win a majority of districts is $1/2\cdot4/7 = 2/7$.

Extra credit
------------
Based on numerical optimization, my guess is that to maximize the probability of winning 5 districts
means $P = 1/2$ for all districts, which gives $L_A = 1/2$,
and to maximize the probability of winning 4 districts means $P = 7/8$ for 4 districts
and $P = 0$ for the other three, which gives $L_B = 1695/4096$.

That makes $L_A-L_B = 353/4096 \approx 0.086$.

In [1]:
p = [var(f"P_{i}") for i in range(7)]

def win_probability(v, can_lose):
    if can_lose <= 0:
        return product(v)
    elif can_lose >= len(v):
        return 1
    else:
        return v[0]*win_probability(v[1:], can_lose) + (1-v[0])*win_probability(v[1:], can_lose-1)

In [2]:
win5 = win_probability(p, 2).subs(p[0] == 7/2 - sum(p[1:]))
win4 = win_probability(p, 3).subs(p[0] == 7/2 - sum(p[1:]))
dummymander = win_probability([1-v for v in p], 3).subs(p[0] == 7/2 - sum(p[1:]))

In [3]:
constraints = p[1:] + [1 - v for v in p[1:]] + [7/2 - sum(p[1:]), sum(p[1:]) - 5/2] # all 7 probabilities are between 0 and 1

In [4]:
def t(objective, init):
    s = minimize_constrained(-objective, constraints, init)
    print(s)
    print(objective.subs([p[i+1] == s[i] for i in [0..5]]),
          numerical_approx(objective.subs([p[i+1] == init[i] for i in [0..5]])),
          objective.subs([p[i+1] == init[i] for i in [0..5]]))
    print(dummymander.subs([p[i+1] == s[i] for i in [0..5]]),
          numerical_approx(dummymander.subs([p[i+1] == init[i] for i in [0..5]])),
          dummymander.subs([p[i+1] == init[i] for i in [0..5]]))

Trying spreading the probability equally among 7, 6, 5, and 4 districts, while allocating none
to the remaining districts, and numerically maximizing the probability of winning at least 4 districts
shows that 7 and 5 are local maxima, and 4 looks like the maximum.

In [5]:
t(win4, [1/2,1/2,1/2,1/2,1/2,1/2])

(0.5, 0.5, 0.5, 0.5, 0.5, 0.5)
0.5 0.500000000000000 1/2
0.5 0.500000000000000 1/2


In [6]:
t(win4, [7/12,7/12,7/12,7/12,7/12,7/12])

(0.8750705937498171, 0.87508590411377, 0.8749716361632973, -1.4211737808035328e-20, 9.044027311660852e-22, 3.002624797434899e-22)
0.5861816292988414 0.509793086634088 761117/1492992
0.41381837070115834 0.490206913365912 731875/1492992


In [7]:
t(win4, [7/10,7/10,7/10,7/10,7/10,0])

(0.6998526298946083, 0.7000911548167215, 0.7000034435301269, 0.7000828472459925, 2.5673186676929648e-21, 0.6999699245125504)
0.5282199981478016 0.528220000000000 26411/50000
0.47178000185219826 0.471780000000000 23589/50000


In [8]:
t(win4, [7/8,7/8,7/8,7/8,0,0])

(0.8749857705606411, 0.8749582709331608, 0.8750883698434949, 0.8749675886627035, 3.0285547259267025e-21, 3.2417933845703016e-21)
0.5861816364894484 0.586181640625000 2401/4096
0.4138183635105515 0.413818359375000 1695/4096


Trying spreading the probability equally among 7, 6, and 5 districts, while allocating none
to the remaining districts, and numerically maximizing the probability of winning at least 5 districts
shows that 7 looks like the maximum.

In [9]:
t(win5, [1/2,1/2,1/2,1/2,1/2,1/2])

(0.5000080846660754, 0.4999540362775583, 0.49996591573741644, 0.5000271883729279, 0.5000668898195724, 0.49996196341428323)
0.22656249919894245 0.226562500000000 29/128
0.5000000000000111 0.500000000000000 1/2


In [10]:
t(win5, [7/10,7/10,7/10,7/10,7/10,0])

(0.4999307803871671, 0.500061830174334, 0.4999152273997585, 0.5000627104339547, 0.4999337969900073, 0.5000361220044268)
0.2265624977371128 0.168070000000000 16807/100000
0.4999999999999591 0.471780000000000 23589/50000


In [11]:
t(win5, [7/12,7/12,7/12,7/12,7/12,7/12])

(0.4999019486797972, 0.5000213539980374, 0.49985495386218265, 0.49987789781438324, 0.5000824721323774, 0.4999811637296331)
0.2265624897112633 0.208259320880487 621859/2985984
0.500000000001396 0.490206913365912 731875/1492992


In [12]:
L_A = 1/2
L_B = 1695/4096
L_A-L_B, numerical_approx(L_A-L_B)

(353/4096, 0.0861816406250000)